## Creating Silver Fact Tables Using Spark

Transforming bronze fact tables into clean, deduplicated silver tables using PySpark:
* **slv_order_items** - Order line items with data quality fixes
* **slv_order_returns** - Order returns with standardized reasons
* **slv_order_shipments** - Order shipments with carrier details

**Key Features**:
- PySpark transformations with window functions
- Delta Lake MERGE for upserts (idempotent)
- Data quality transformations and deduplication

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Configuration
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

print(f"Using catalog: {catalog_name}")

In [0]:
# Order items - cleaning and transformation

df_order_items = spark.table(f"{catalog_name}.bronze.brz_order_items")

# Define window for deduplication
window_order_items = Window.partitionBy("order_id", "item_seq").orderBy(F.col("_ingested_at").desc())

# Apply transformations
df_order_items_clean = (df_order_items
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("quantity",
        F.when(F.col("quantity") == "Two", 2)
         .otherwise(F.col("quantity").cast("int"))
    )
    .withColumn("unit_price_currency", F.upper(F.trim(F.col("unit_price_currency"))))
    .withColumn("unit_price", F.regexp_replace(F.col("unit_price"), "[$]", "").cast("double"))
    .withColumn("discount_pct", F.regexp_replace(F.col("discount_pct"), "%", "").cast("double"))
    .withColumn("channel",
        F.when(F.col("channel") == "web", "Website")
         .when(F.col("channel") == "app", "Mobile")
         .otherwise(F.col("channel"))
    )
    .withColumn("coupon_code", F.lower(F.trim(F.col("coupon_code"))))
    .withColumn("rn", F.row_number().over(window_order_items))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_order_items"):
    delta_order_items = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_order_items")
    
    (delta_order_items.alias("target")
        .merge(
            df_order_items_clean.alias("source"),
            "target.order_id = source.order_id AND target.item_seq = source.item_seq"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_order_items_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_order_items")

In [0]:
# Order returns - cleaning and transformation

df_order_returns = spark.table(f"{catalog_name}.bronze.brz_order_returns")

# Define window for deduplication
window_order_returns = Window.partitionBy("order_id").orderBy(F.col("return_ts").desc())

# Apply transformations
df_order_returns_clean = (df_order_returns
    .withColumn("reason",
        F.when(F.trim(F.col("reason")) == "", "Not Specified")
         .otherwise(F.initcap(F.trim(F.col("reason"))))
    )
    .withColumn("rn", F.row_number().over(window_order_returns))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_order_returns"):
    delta_order_returns = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_order_returns")
    
    (delta_order_returns.alias("target")
        .merge(
            df_order_returns_clean.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_order_returns_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_order_returns")

In [0]:
# Order shipments - cleaning and transformation

df_order_shipments = spark.table(f"{catalog_name}.bronze.brz_order_shipments")

# Define window for deduplication
window_order_shipments = Window.partitionBy("shipment_id").orderBy(F.col("_ingested_at").desc())

# Apply transformations
df_order_shipments_clean = (df_order_shipments
    .withColumn("shipment_id", F.trim(F.col("shipment_id")))
    .withColumn("carrier", F.trim(F.col("carrier")))
    .withColumn("rn", F.row_number().over(window_order_shipments))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_order_shipments"):
    delta_order_shipments = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_order_shipments")
    
    (delta_order_shipments.alias("target")
        .merge(
            df_order_shipments_clean.alias("source"),
            "target.shipment_id = source.shipment_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_order_shipments_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_order_shipments")

In [0]:
# Summary of silver fact tables

print("Silver Fact Tables Summary")

tables = ["slv_order_items", "slv_order_returns", "slv_order_shipments"]

for table in tables:
    count = spark.table(f"{catalog_name}.silver.{table}").count()
    print(f"{table:25} : {count:,} rows")

print("\n All silver fact tables processed successfully!")